## ProofHound — train the Harness model (QLoRA) on Colab

Runs `training/train/train_local.py` from the repo, so Colab and the laptop train the same way and
the adapter matches `training/serve/serve_local.py` exactly (shared format in
`src/proofhound/core/harness/chat_format.py`).

**Before you start**
1. Runtime → Change runtime type → **GPU** (T4 16 GB runs 3B with the default settings or 7B with
   the reduced-rank / attention-only settings in the cell below; L4/A100 are faster and use bf16).
2. Put the training set in Google Drive at `MyDrive/proofhound/data/`: `harness_train.jsonl`, built on
   the laptop by `training/generate/build_dataset.py` from the distillation logs (only harnesses whose
   crash was confirmed, prompts in the current format). `training/data/` is gitignored, so it is not
   in the clone.
3. If the GitHub repo is private, add a Colab secret named `GITHUB_TOKEN` (key icon on the left).

Checkpoints are written to Drive every `SAVE_STEPS` steps. If the session disconnects, run all
cells again — training resumes from the last checkpoint.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Settings — choose 3B or 7B, then run all cells
#
# 3B (safe default, any T4):
#   MODEL      = "Qwen/Qwen2.5-Coder-3B-Instruct"
#   MAX_LENGTH = 3072
#   LORA_RANK  = 16
#   TARGET_MODULES = ""          # "" → all-linear
#
# 7B (fits a free T4 with these reduced settings):
#   MODEL      = "Qwen/Qwen2.5-Coder-7B-Instruct"
#   MAX_LENGTH = 2048
#   LORA_RANK  = 8
#   TARGET_MODULES = "q_proj k_proj v_proj o_proj"   # attention-only keeps MLP grads out of VRAM

MODEL          = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_LENGTH     = 2048
LORA_RANK      = 8
TARGET_MODULES = "q_proj k_proj v_proj o_proj"

EPOCHS     = 3
SAVE_STEPS = 25
DRIVE_DIR  = "/content/drive/MyDrive/proofhound"
# A new folder per model: training resumes from any checkpoint already in OUT.
OUT = f"{DRIVE_DIR}/adapters/{MODEL.split('/')[-1]}-harness-v2"
REPO_URL = "github.com/moabdelmoati/proofhound.git"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"
if not os.path.isdir("/content/proofhound"):
    !git clone -q --depth 1 {url} /content/proofhound
%cd /content/proofhound
!git log -1 --oneline

In [ ]:
!pip -q install -e . "transformers==5.20.0" "peft==0.21.2" bitsandbytes accelerate

In [ ]:
import glob, shutil
os.makedirs("training/data", exist_ok=True)
for f in glob.glob(f"{DRIVE_DIR}/data/*.jsonl") + glob.glob(f"{DRIVE_DIR}/*.jsonl"):
    shutil.copy(f, "training/data/")
DATA = "training/data/harness_train.jsonl"
print("data files:", os.listdir("training/data"))
assert os.path.exists(DATA), f"upload harness_train.jsonl to {DRIVE_DIR}/data/"

## 1. Inspect the dataset (no GPU work)
Check the counts: **stale_prompt_version must be 0**, otherwise the model is trained on prompts the
pipeline no longer sends. The printout also shows one rendered prompt and the target the loss is computed on.

In [ ]:
!python training/train/train_local.py --model {MODEL} --max-length {MAX_LENGTH} --data {DATA} --dry-run

## 2. Train
Saves the adapter (and `proofhound_format.json`) to `OUT` on Drive.

In [ ]:
_target_flag = f"--target-modules {TARGET_MODULES}" if TARGET_MODULES else ""
!python training/train/train_local.py \
  --model {MODEL} \
  --max-length {MAX_LENGTH} \
  --lora-rank {LORA_RANK} \
  {_target_flag} \
  --epochs {EPOCHS} \
  --save-steps {SAVE_STEPS} \
  --data {DATA} \
  --out "{OUT}"

## 3. Smoke test
Generates one harness, exactly as the local server would, for a benchmark prompt the model never
trained on (`harness_attempts.jsonl` holds eval-split records only — upload it to the same folder).

In [ ]:
!python training/serve/generate_test.py --adapter "{OUT}" --device cuda --load-4bit --prompts training/data/harness_attempts.jsonl

## 4. Download the adapter
Unzip into `models/harness-adapter/` on the laptop, then:
`python training/serve/serve_local.py --adapter models/harness-adapter --device cuda --load-4bit`

In [ ]:
import shutil
from google.colab import files
for checkpoint in glob.glob(f"{OUT}/checkpoint-*"):
    shutil.rmtree(checkpoint)
archive = shutil.make_archive("/content/harness-adapter", "zip", OUT)
files.download(archive)